In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_4
from src.rl.trainer_team import train_team_ppo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

⚡ Device: cuda


In [13]:
# ── STAGE 4 CONFIGURATION ──
STAGE = 4
TEAM_SIZE = 2  # 2v2 match format
NUM_ENVS = 16
MAX_STEPS = 1800  # 30.0s at 60 Hz
TIME_LIMIT = 30.0

SAVE_DIR = f"models/stage{STAGE}"
POOL_DIR = f"models/stage{STAGE}/pool"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(POOL_DIR, exist_ok=True)

In [14]:
def make_env(env_idx: int):

  def _init():
    import torch

    torch.set_num_threads(1)

    # 50/50 balance between Red and Blue teams across environments
    is_red = env_idx % 2 == 0
    learner_team = "red" if is_red else "blue"
    opp_team = "blue" if is_red else "red"

    opp_ctrl = PoolController(pool_dir=POOL_DIR, team=opp_team, device="cpu", heuristic_pct=1.0)

    # Build 2v2 Roster (2 Learners vs 2 Opponents)
    roster = []
    for i in range(TEAM_SIZE):
      roster.append(
          PlayerSlot(
              learner_team,
              PlayerStats(name=f"Learner_{i}", accel=3200.0),
              controller="RL",
          )
      )
    for i in range(TEAM_SIZE):
      roster.append(
          PlayerSlot(
              opp_team,
              PlayerStats(name=f"Opponent_{i}", accel=3200.0),
              controller=opp_ctrl,
          )
      )

    cfg = MatchConfig(
        mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99),
        roster=roster,
    )

    return MatchEnv(
        match_config=cfg,
        reward_shaper=DenseReward_4(team=learner_team),
        reset_strategy=RandomReset(),
        learner_team=learner_team,
        max_steps=MAX_STEPS,
    )

  return _init


# Parallelize 16 environments across CPU cores
train_envs = gym.vector.AsyncVectorEnv(
    [make_env(i) for i in range(NUM_ENVS)], context="spawn"
)

# Unified 80-dim policy network
model = ActorCritic(obs_dim=80).to(device)

# Seed Stage 4 using the best Stage 3 weights
stage3_best = "models/stage3/best_model.pt"
if os.path.exists(stage3_best):
  model.load_state_dict(
      torch.load(stage3_best, map_location=device, weights_only=False)
  )
  print(f"✅ Bootstrapping Stage 4 with Champion weights from: {stage3_best}")

✅ Bootstrapping Stage 4 with Champion weights from: models/stage3/best_model.pt


In [ ]:
# Multi-Agent IPPO Training Loop
train_team_ppo(
    envs=train_envs,
    model=model,
    device=device,
    team_size=TEAM_SIZE,
    double_eval=False,
    total_timesteps=3_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
)

train_envs.close()

🚀 Multi-Agent Training (IPPO) | Format: 2v2 | Envs: 16 | Batch: 8192


In [18]:
from src.rl.evaluator import evaluate_and_generate_html_2

evaluate_and_generate_html_2(
    red_agent="models/stage4/best_model.pt",
    blue_agent="heuristic",
    team_size=2,
    num_episodes=3,
    filename="stage4_2v2_vs_heuristic.html",
)

🎬 Multi-Agent Replay generated: /home/minh-quan/Documents/Haxball project/training/render/stage4_2v2_vs_heuristic.html


'render/stage4_2v2_vs_heuristic.html'